In [60]:
%matplotlib widget

import pickle

import matplotlib.pyplot as plt
import numpy as np
from larch import Group
from larch.io import read_ascii, read_xdi  # noqa
from larch.xafs import autobk, mback, pre_edge  # noqa
from pymatgen.analysis.chemenv.coordination_environments.chemenv_strategies import (
    MultiWeightsChemenvStrategy,
)
from pymatgen.analysis.chemenv.coordination_environments.coordination_geometry_finder import (
    LocalGeometryFinder,
)
from pymatgen.analysis.chemenv.coordination_environments.structure_environments import (
    LightStructureEnvironments,
)
from pymatgen.core.structure import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

from xasml import resource_path

plt.style.use("../../models/stylelib/presentation.mplstyle")

In [61]:
# Reference compounds (ranks 1-17 plus 19, 20, 22 of the iron_coordination_environments
# table, keyed by mineral name): the absorbing-Fe formula, the XDI scan, and the CIF
# passed to ChemEnv to determine the local Fe coordination environment.
COMPOUNDS = {
    "Chalcopyrite": {
        "formula": "CuFeS2",
        "xdi": "XASDB/Chalcopyrite/Chalcopyrite_idx9912a.dat",
        "cif": "COD/Chalcopyrite/cod_1010940.cif",
    },
    "Wustite": {
        "formula": "FeO",
        "xdi": "XASDB/Wustite/Wustite_id6cn8xf.dat",
        "cif": "COD/Wustite/cod_1011169.cif",
    },
    "Andradite": {
        "formula": "Ca3Fe2(SiO4)3",
        "xdi": "XASDB/Andradite/Andradite_idemsvvd.dat",
        "cif": "COD/Andradite/cod_2101484.cif",
    },
    "Siderite": {
        "formula": "Fe(CO3)",
        "xdi": "XASDB/Siderite/Siderite_iddujsej.dat",
        "cif": "COD/Siderite/cod_2104746.cif",
    },
    "Rodolicoite": {
        "formula": "Fe(PO4)",
        "xdi": "XASDB/Iron (III) Phosphate/Iron (III) Phosphate_id4ln8pg.dat",
        "cif": "COD/Iron (III) Phosphate/cod_9012512.cif",
    },
    "Hedenbergite": {
        "formula": "CaFe(Si2O6)",
        "xdi": "XASDB/Hedenbergite/Hedenbergite_idom107e.dat",
        "cif": "COD/Hedenbergite/cod_9000336.cif",
    },
    "Hematite": {
        "formula": "a-Fe2O3",
        "xdi": "XASDB/Hematite/Hematite_idal8hze.dat",
        "cif": "COD/Hematite/cod_1532119.cif",
    },
    "Pyrite": {
        "formula": "FeS2",
        "xdi": "XASLIB/FeS2_rt_01.xdi",
        "cif": "COD/Pyrite/cod_1544891.cif",
    },
    "Scorodite": {
        "formula": "Fe(AsO4).2H2O",
        "xdi": "XASDB/Scorodite/Scorodite_id5bjue8.dat",
        "cif": "COD/Scorodite/cod_2212542.cif",
    },
    "Wolframite": {
        "formula": "Fe(WO4)",
        "xdi": "XASDB/Wolframite/Wolframite_idlxnsgy.dat",
        "cif": "COD/Wolframite/cod_9000223.cif",
    },
    "Aegirine": {
        "formula": "NaFe(Si2O6)",
        "xdi": "XASDB/Aegerine/Aegerine_id9pyqfv.dat",
        "cif": "COD/Aegerine/cod_9000327.cif",
    },
    "Humboldtine": {
        "formula": "Fe(C2O4).2H2O",
        "xdi": "XASDB/Iron (II) Oxalate/Iron (II) Oxalate_idng95ys.dat",
        "cif": "COD/Iron (II) Oxalate/cod_9017265.cif",
    },
    "Ilmenite": {
        "formula": "FeTiO3",
        "xdi": "XASDB/Ilmenite/Ilmenite_idextn4w.dat",
        "cif": "COD/Ilmenite/cod_1011033.cif",
    },
    "Goethite": {
        "formula": "a-FeOOH",
        "xdi": "XASDB/Goethite/Goethite_idghzdol.dat",
        "cif": "COD/Goethite/cod_2211652.cif",
    },
    "Heterosite": {
        "formula": "Fe(PO4) (heterosite)",
        "xdi": "XASDB/Iron Phosphate/Iron Phosphate_idux4tlj.dat",
        "cif": "COD/Iron Phosphate/cod_9015219.cif",
    },
    "Scorzalite": {
        "formula": "FeAl2(PO4)2(OH)2",
        "xdi": "XASDB/Scorzalite/Scorzalite_idcgeh2i.dat",
        "cif": "COD/Scorzalite/cod_9007451.cif",
    },
    "Lepidocrocite": {
        "formula": "g-FeOOH",
        "xdi": "XASDB/Lepidocrocite/Lepidocrocite_idj58orc.dat",
        "cif": "COD/Lepidocrocite/cod_1011026.cif",
    },
}


In [62]:
# Determine the Fe coordination environment of every compound with ChemEnv, using the
# same configuration as the dataset build script
# (scripts/datasets/fdmnes/build_materials_database.py). The continuous symmetry
# measure (CSM) of each environment is the deviation from the idealised geometry, with
# 0 meaning a perfect match.
STRUCTURE_ENVIRONMENT_CONFIGURATION = {
    "min_cn": 4,
    "max_cn": 6,
    "only_symbols": ["S:4", "T:4", "T:5", "S:5", "O:6", "T:6"],
}
STRATEGY = MultiWeightsChemenvStrategy.stats_article_weights_parameters()

coordination = {}
for compound, info in COMPOUNDS.items():
    cif = resource_path(f"xasml:datasets/experimental/databases/{info['cif']}")
    structure = Structure.from_file(cif)

    spa = SpacegroupAnalyzer(structure, symprec=0.01, angle_tolerance=5)
    equivalent = sorted(set(spa.get_symmetry_dataset().equivalent_atoms))
    # Use site.species (Composition) rather than site.specie so disordered sites,
    # e.g. (Fe,Mg) mixing in carpholite, are also picked up.
    fe_indices = [
        i
        for i in equivalent
        if any(el.symbol == "Fe" for el in structure.sites[i].species)
    ]

    lgf = LocalGeometryFinder()
    lgf.setup_parameters(centering_type="standard")
    lgf.setup_structure(structure)
    se = lgf.compute_structure_environments(
        only_indices=fe_indices, **STRUCTURE_ENVIRONMENT_CONFIGURATION
    )
    lse = LightStructureEnvironments.from_structure_environments(
        strategy=STRATEGY, structure_environments=se
    )
    coordination[compound] = [(i, lse.coordination_environments[i]) for i in fe_indices]  # type: ignore

In [ ]:
PRE_EDGE_DEFAULTS = {
    "pre1": -150,
    "pre2": -30,
    "norm1": 150,
    "norm2": 950,
    "nnorm": 2,
    "nvict": 2,
}
# Two-class palette shared with the rest of the manuscript figures.
CLASS_COLORS = {"O:6": "#832db6", "T:4": "#f97b72"}
CLASS_LABELS = {"O:6": r"$O_h$", "T:4": r"$T_d$"}
FE_K_EDGE = 7112.0

XMIN, XMAX = 7100, 7180
OFFSET = 0.7  # vertical step between stacked spectra
LABEL_PAD = 1.0  # eV from the right spine to the start of the label

fig, ax = plt.subplots(figsize=(6.3, 8.0))
data = {}
eshifts = {}
for i, (compound, info) in enumerate(COMPOUNDS.items()):
    path = resource_path(f"xasml:datasets/experimental/databases/{info['xdi']}")
    dat = read_ascii(path)
    dat.mu = -np.log(dat.itrans / dat.i0)

    # Align the energy axis on the reference foil, which sits downstream of the Itrans
    # detector, so its incident flux is Itrans. Scans without a foil are left unshifted.
    if "irefer" in dat.array_labels:
        ref = Group(energy=dat.energy.copy(), mu=np.log(dat.itrans / dat.irefer))
        pre_edge(ref)
        eshift = FE_K_EDGE - ref.e0  # type: ignore
        dat.energy = dat.energy + eshift
        eshifts[compound] = (float(ref.e0), float(eshift))  # type: ignore
    else:
        eshifts[compound] = None

    # Standard XANES normalisation: Victoreen (order 2) pre-edge subtraction and a
    # quadratic post-edge fit, with the slope flattened out (dat.flat). MBACK is
    # avoided because its match to a pure-Fe atomic background biases sulfides.
    pre_edge(dat, **PRE_EDGE_DEFAULTS)

    # Dominant environment of the first unique Fe site, used to colour the curve.
    symmetry = coordination[compound][0][1][0]["ce_symbol"]

    x = dat.energy
    y = dat.flat
    yo = y + i * OFFSET

    # Plot only the data window; label outside the right spine.
    mask = (x >= XMIN) & (x <= XMAX)
    ax.plot(x[mask], yo[mask], lw=1, color=CLASS_COLORS[symmetry])
    ax.text(
        XMAX + LABEL_PAD,
        yo[mask][-1],
        compound,
        va="center",
        fontsize=10,
        clip_on=False,
    )

    data[compound] = (x, y, symmetry)

# Class legend placed in the empty top-left corner of the axes.
handles = [
    plt.Line2D([0], [0], color=CLASS_COLORS[s], lw=1.4, label=CLASS_LABELS[s])
    for s in ("O:6", "T:4")
]
ax.legend(handles=handles, loc="upper left", frameon=False)

# ax.grid(True, color="0.92", linewidth=0.6, zorder=0, axis="x")
ax.set_axisbelow(True)
ax.tick_params(
    direction="in", length=4, top=False, right=False, left=False, labelleft=False
)
ax.set_xlim(XMIN, XMAX)
ax.set_ylim(-0.2, (len(COMPOUNDS) - 1) * OFFSET + 2.0)
ax.set_xlabel("Energy (eV)")
for side in ("left", "right", "top"):
    ax.spines[side].set_visible(False)
fig.tight_layout()
fig.subplots_adjust(right=0.75)

In [72]:
# Local coordination environment of the absorbing Fe atom in each reference compound,
# as determined by ChemEnv, together with the energy-axis alignment shift derived from
# the reference foil ("-" if no reference channel is present in the scan).
header = (
    f"{'Compound':18s} {'Formula':28s} {'RefEdge (eV)':>14s} "
    f"{'Shift (eV)':>11s} {'Site':>5s} {'Symbol':>7s} {'Fraction':>9s} {'CSM':>7s}"
)
print(header)
print("-" * len(header))
for compound, sites in coordination.items():
    formula = COMPOUNDS[compound]["formula"]
    shift_info = eshifts.get(compound)
    if shift_info is None:
        ref_str = f"{'-':>14s}"
        shift_str = f"{'-':>11s}"
    else:
        ref_e0, shift = shift_info
        ref_str = f"{ref_e0:>14.3f}"
        shift_str = f"{shift:>+11.3f}"
    for site_index, environments in sites:
        for env in environments:
            print(
                f"{compound:18s} {formula:28s} {ref_str} {shift_str} "
                f"{site_index:>5d} {env['ce_symbol']:>7s} "
                f"{env['ce_fraction']:>9.3f} {env['csm']:>7.3f}"
            )

Compound           Formula                        RefEdge (eV)  Shift (eV)  Site  Symbol  Fraction     CSM
----------------------------------------------------------------------------------------------------------
Chalcopyrite       CuFeS2                             7112.000      +0.000     0     T:4     1.000   0.011
Wustite            FeO                                7111.591      +0.409     0     O:6     1.000   0.000
Andradite          Ca3Fe2(SiO4)3                      7111.595      +0.405    24     O:6     1.000   0.014
Siderite           Fe(CO3)                            7112.000      +0.000     0     O:6     1.000   0.066
Rodolicoite        Fe(PO4)                            7112.000      +0.000     0     T:4     1.000   0.187
Hedenbergite       CaFe(Si2O6)                        7111.595      +0.405     4     O:6     1.000   0.265
Hematite           a-Fe2O3                            7112.000      +0.000     0     O:6     1.000   0.288
Pyrite             FeS2              

In [73]:
path = resource_path("xasml:datasets/experimental/normalized_experimental_data.pkl")
with open(path, "wb") as f:
    pickle.dump(data, f)